In [ ]:
import pandas as pd
import numpy as np
import wget

Download the cross verified database from here: https://data.sciencespo.fr/file.xhtml?fileId=4432&version=2.2 
and unzip it into a folder called data

In [ ]:
# use wget to download the FCDO Geographical Names Index
url = "https://assets.publishing.service.gov.uk/media/65fd8475f1d3a0001132adf4/FCDO_Geographical_Names_Index_March_2024.csv"
output_path = (
    "data/FCDO_Geographical_Names_Index_March_2024.csv"  # Optional: specify output path
)
try:
    filename = wget.download(url, out=output_path)
    print(f"File downloaded: {filename}")
except Exception as e:
    print(f"Error downloading file: {e}")

# Load the FCDO data in using pandas
fcdo = pd.read_csv(
    "data/FCDO_Geographical_Names_Index_March_2024.csv", encoding_errors="ignore"
)

# Load the cross verified database in using Pandas
cv = pd.read_csv(
    "data/cross-verified-database.csv",
    usecols=["name", "wiki_readers_2015_2018"],
    encoding_errors="ignore",
)
# cf_full = pd.read_csv("cross-verified-database.csv", encoding_errors="ignore") # in case you want the full dataset other than the columns of interest

In [ ]:
# make all names lowercase for string matching and split them into columns using the _ separater
names = cv
names["name"] = cv["name"].str.lower()
names[["first", "last"]] = names.name.str.rsplit("_", n=1, expand=True)

In [ ]:
# create a countries list that is lowercase for string matching
countries = fcdo["Name"]
countries = countries.str.lower()

In [ ]:
# write a function to add every letter in every position of every name on the countries list


def add_every_letter(text):
    result = []
    for i in range(len(text) + 1):
        for char in "abcdefghijklmnopqrstuvwxyz":
            new_text = text[:i] + char + text[i:]
            result.append(new_text)
    return result

In [ ]:
result = [add_every_letter(x) for x in countries]
df = pd.DataFrame(result)
countries_permutations = pd.concat([df[col] for col in df.columns])

In [ ]:
test = names["first"].isin(countries_permutations)

In [ ]:
# check which first names from the CV dataset match those generated in the countries_permutations list
first = countries_permutations.isin(names["first"])
first = countries_permutations[first]
first = pd.DataFrame(first).rename(columns={0: "first_name"})
# first["match_first"] = 'Y'
print(first)

In [ ]:
# check which first names from the CV dataset match those generated in the countries_permutations list
last = countries_permutations.isin(names["last"])
last = countries_permutations[last]
last = pd.DataFrame(last).rename(columns={0: "last_name"}).dropna()
# last["match_last"] = 'Y'
print(last)

In [ ]:
# merge the first and last names lists
df_merge = names.merge(first, left_on="first", right_on="first_name", how="left")
df_merge = df_merge.merge(last, left_on="last", right_on="last_name", how="left")

In [ ]:
# set a Match flag on the final dataset that will indicate if there is a match between the first or last name and the countries_permutations list
final_prep = df_merge
final_prep["match"] = None
final_prep.loc[
    final_prep["first_name"].notna() | final_prep["last_name"].notna(), "match"
] = "Y"
final = final_prep.dropna(subset=["match"])

The final dataset first and last names of those who matchd and their wikipedia readership number, which can be taken as a measure of popularity. Sort and figure out which ones you want to use at your leisure. 